In [ ]:
import argparse
import pathlib
import string

from esm import Alphabet, FastaBatchedDataset, ProteinBertModel, pretrained, MSATransformer
import pandas as pd
from tqdm import tqdm

import itertools
from typing import List, Tuple
import numpy as np
import torch
import os
import numpy as np
import pandas as pd
import tqdm.notebook
from scipy.special import softmax
import argparse

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# curl https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t36_3B_UR50D.pt -o esm2_t36_3B_UR50D.pt
# curl https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t48_15B_UR50D.pt -o esm2_t48_15B_UR50D.pt


# 1. Allow argparse.Namespace to be unpickled
torch.serialization.add_safe_globals([argparse.Namespace])

# 2. Define your path
model_path = "/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_proteome/00.esm_cache/esm2_t36_3B_UR50D.pt"

# 3. Load locally (this should now work)
model, alphabet = pretrained.load_model_and_alphabet_local(model_path)
batch_converter = alphabet.get_batch_converter()

model.eval()
model.to(DEVICE)
print("ESM-2 Loaded from local file.")

In [ ]:
def get_masked_marginal_scores(sequence, model, alphabet, batch_size=4):
    from torch.nn.functional import log_softmax

    model.eval()
    batch_converter = alphabet.get_batch_converter()
    device = next(model.parameters()).device

    standard_aa = list("ACDEFGHIKLMNPQRSTVWY")
    aa_indices = torch.tensor([alphabet.get_idx(a) for a in standard_aa]).to(device)

    # Tokenize WT once
    data = [("wt", sequence)]
    _, _, wt_tokens = batch_converter(data)
    wt_tokens = wt_tokens.to(device)

    seq_len = len(sequence)

    masked_tokens = []
    for i in range(seq_len):
        tokens = wt_tokens.clone()
        tokens[0, i + 1] = alphabet.mask_idx  # mask at token level
        masked_tokens.append(tokens)

    scores = []

    with torch.no_grad():
        for i in range(0, seq_len, batch_size):
            batch = torch.cat(masked_tokens[i:i+batch_size], dim=0)

            logits = model(batch)["logits"]
            log_probs = log_softmax(logits, dim=-1)

            for j in range(batch.size(0)):
                pos = i + j
                lp = log_probs[j, pos + 1, aa_indices]
                scores.append(lp.cpu().numpy())

    score_matrix = np.vstack(scores)

    final_scores = []
    for k in range(seq_len):
        wt_aa = sequence[k]
        wt_idx = standard_aa.index(wt_aa)
        final_scores.append(score_matrix[k] - score_matrix[k, wt_idx])

    df = pd.DataFrame(final_scores, columns=standard_aa)
    df.insert(0, "WT", list(sequence))
    df.insert(0, "Pos", range(1, seq_len + 1))

    return df

# 1. Define your sequence (Example: GFP)
test_seq = "MLDDRARMEAAKKEKVEQILAEFQLQEEDLKKVMRRMQKEMDRGLRLETHEEASVKMLPTYVRSTPEGSEVGDFLSLDLGGTNFRVMLVKVGEGEEGQWSVKTKHQMYSIPEDAMTGTAEMLFDYISECISDFLDKHQMKHKKLPLGFTFSFPVRHEDIDKGILLNWTKGFKASGAEGNNVVGLLRDAIKRRGDFEMDVVAMVNDTVATMISCYYEDHQCEVGMIVGTGCNACYMEEMQNVELVEGDEGRMCVNTEWGAFGDSGELDEFLLEYDRLVDESSANPGQQLYEKLIGGKYMGELVRLVLLRLVDENLLFHGEASEQLRTRGAFETRFVSQVESDTGDRKQIYNILSTLGLRPSTTDCDIVRRACESVSTRAAHMCSAGLAGVINRMRESRSEDVMRITVGVDGSVYKLHPSFKERFHASVRRLTPSCEITFIESEEGSGRGAALVSAVACKKACMLGQ"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores = get_masked_marginal_scores(test_seq, model, alphabet, batch_size=2)

df_scores

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_heatmap(df, start_pos=1, end_pos=None):
    """
    Visualizes the mutation scores as a heatmap.
    Handles removing the text columns 'WT' and 'Pos' automatically.
    """
    # 1. Prepare Data
    plot_df = df.copy()
    
    # Filter by position (optional slicing)
    if end_pos is not None:
        plot_df = plot_df[(plot_df["Pos"] >= start_pos) & (plot_df["Pos"] <= end_pos)]
    
    # Create a nice label for the x-axis: "Position (WildType)" -> e.g., "1 (M)"
    plot_df['Label'] = plot_df['Pos'].astype(str) + " (" + plot_df['WT'] + ")"
    
    # Set this label as the index (so it becomes the x-axis tick labels)
    plot_df = plot_df.set_index('Label')
    
    # Drop the original metadata columns so only numeric scores remain
    # We check if they exist first to avoid errors if run multiple times
    cols_to_drop = [c for c in ['Pos', 'WT'] if c in plot_df.columns]
    heatmap_data = plot_df.drop(columns=cols_to_drop)
    
    # Transpose so Mutations are on Y-axis, Positions on X-axis
    heatmap_data = heatmap_data.T

    # 2. Plotting
    plt.figure(figsize=(20, 8)) 
    
    sns.heatmap(
        heatmap_data, 
        cmap="bwr_r",     # Blue=Good, White=Neutral, Red=Bad
        center=0, 
        vmin=-15,         # Clamp very negative scores to keep colors visible
        vmax=5,
        cbar_kws={'label': 'Log-Likelihood Ratio (LLR)'},
        linewidths=0.5,   # Grid lines make it easier to read
        linecolor='lightgray'
    )
    
    plt.title(f"ESM-2 Variant Effect Landscape")
    plt.xlabel("Position (Wild Type)")
    plt.ylabel("Mutant Amino Acid")
    
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Run the fixed function
plot_heatmap(df_scores)

In [ ]:
df_scores.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/gck_esm2_3b.csv", index=False)

In [ ]:
ptp1b = "MEMEKEFEQIDKSGSWAAIYQDIRHEASDFPCRVAKLPKNKNRNRYRDVSPFDHSRIKLHQEDNDYINASLIKMEEAQRSYILTQGPLPNTCGHFWEMVWEQKSRGVVMLNRVMEKGSLKCAQYWPQKEEKEMIFEDTNLKLTLISEDIKSYYTVRQLELENLTTQETREILHFHYTTWPDFGVPESPASFLNFLFKVRESGSLSPEHGPVVVHCSAGIGRSGTFCLADTCLLLMDKRKDPSSVDIKKVLLEMRKFRMGLIQTADQLRFSYLAVIEGAKFIMGDSSVQDQWKELSHEDLEPPPEHIPPPPRPPKRILEPHNGKCREFFPNHQWVKEETQEDKDCPIKEEKGSPLNAAPYGIESMSQDTEVRSRVVGGSLRGAQAASPAKGEPSLPEKDEDHALSYWKPFLVNMCVATVLTAGAYLCYRFLFNSNT"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores2 = get_masked_marginal_scores(ptp1b, model, alphabet, batch_size=2)
df_scores2.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/ptp1b_esm2_3b.csv", index=False)

In [ ]:
pdk1 = "MARTTSQLYDAVPIQSSVVLCSCPSPSMVRTQTESSTPPGIPGGSRQGPAMDGTAAEPRPGAGSLQHAQPPPQPRKKRPEDFKFGKILGEGSFSTVVLARELATSREYAIKILEKRHIIKENKVPYVTRERDVMSRLDHPFFVKLYFTFQDDEKLYFGLSYAKNGELLKYIRKIGSFDETCTRFYTAEIVSALEYLHGKGIIHRDLKPENILLNEDMHIQITDFGTAKVLSPESKQARANSFVGTAQYVSPELLTEKSACKSSDLWALGCIIYQLVAGLPPFRAGNEYLIFQKIIKLEYDFPEKFFPKARDLVEKLLVLDATKRLGCEEMEGYGPLKAHPFFESVTWENLHQQTPPKLTAYLPAMSEDDEDCYGNYDNLLSQFGCMQVSSSSSSHSLSASDTGLPQRSGSNIEQYIHDLDSNSFELDLQFSEDEKRLLLEKQAGGNPWHQFVENNLILKMGPVDKRKGLFARRRQLLLTEGPHLYYVDPVNKVLKGEIPWSQELRPEAKNFKTFFVHTPNRTYYLMDPSGNAHKWCRKIQEVWRQRYQSHPDAAVQ"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores3 = get_masked_marginal_scores(pdk1, model, alphabet, batch_size=2)
df_scores3.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/pdk1_esm2_3b.csv", index=False)

In [ ]:
chk1 = "MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAVDCPENIKKEICINKMLNHENVVKFYGHRREGNIQYLFLEYCSGGELFDRIEPDIGMPEPDAQRFFHQLMAGVVYLHGIGITHRDIKPENLLLDERDNLKISDFGLATVFRYNNRERLLNKMCGTLPYVAPELLKRREFHAEPVDVWSCGIVLTAMLAGELPWDQPSDSCQEYSDWKEKKTYLNPWKKIDSAPLALLHKILVENPSARITIPDIKKDRWYNKPLKKGAKRPRVTSGGVSESPSGFSKHIQSNLDFSPVNSASSEENVKYSSSQPEPRTGLSLWDTSPSYIDKLVQGISFSQPTCPDHMLLNSQLLGTPGSSQNPWQRLVKRMTRFFTKLDADKSYQCLKETCEKLGYQWKKSCMNQVTISTTDRRNNKLIFKVNLLEMDDKILVDFRLSKGDGLEFKRHFLKIKGKLIDIVSSQKIWLPAT"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores4 = get_masked_marginal_scores(chk1, model, alphabet, batch_size=2)
df_scores4.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/chk1_esm2_3b.csv", index=False)

In [ ]:
casp1 = "MADKVLKEKRKLFIRSMGEGTINGLLDELLQTRVLNKEEMEKVKRENATVMDKTRALIDSVIPKGAQACQICITYICEEDSYLAGTLGLSADQTSGNYLNMQDSQGVLSSFPAPQAVQDNPAMPTSSGSEGNVKLCSLEEAQRIWKQKSAEIYPIMDKSSRTRLALIICNEEFDSIPRRTGAEVDITGMTMLLQNLGYSVDVKKNLTASDMTTELEAFAHRPEHKTSDSTFLVFMSHGIREGICGKKHSEQVPDILQLNAIFNMLNTKNCPSLKDKPKVIIIQACRGDSPGVVWFKDSVGVSGNLSLPTTEEFEDDAIKKAHIEKDFIAFCSSTPDNVSWRHPTMGSVFIGRLIEHMQEYACSCDVEEIFRKVRFSFEQPDGRAQMPTTERVTLTRCFYLFPGH"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores5 = get_masked_marginal_scores(casp1, model, alphabet, batch_size=2)
df_scores5.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/casp1_esm2_3b.csv", index=False)

In [ ]:
idh1 = "MSKKISGGSVVEMQGDEMTRIIWELIKEKLIFPYVELDLHSYDLGIENRDATNDQVTKDAAEAIKKHNVGVKCATITPDEKRVEEFKLKQMWKSPNGTIRNILGGTVFREAIICKNIPRLVSGWVKPIIIGRHAYGDQYRATDFVVPGPGKVEITYTPSDGTQKVTYLVHNFEEGGGVAMGMYNQDKSIEDFAHSSFQMALSKGWPLYLSTKNTILKKYDGRFKDIFQEIYDKQYKSQFEAQKIWYEHRLIDDMVAQAMKSEGGFIWACKNYDGDVQSDSVAQGYGSLGMMTSVLVCPDGKTVEAEAAHGTVTRHYRMYQKGQETSTNPIASIFAWTRGLAHRAKLDNNKELAFFANALEEVSIETIEAGFMTKDLAACIKGLPNVQRSDYLNTFEFMDKLGENLKIKLAQAKL"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores6 = get_masked_marginal_scores(idh1, model, alphabet, batch_size=2)
df_scores6.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/idh1_esm2_3b.csv", index=False)

In [1]:
import argparse
import pathlib
import string

from esm import Alphabet, FastaBatchedDataset, ProteinBertModel, pretrained, MSATransformer
import pandas as pd
from tqdm import tqdm

import itertools
from typing import List, Tuple
import numpy as np
import torch
import os
import numpy as np
import pandas as pd
import tqdm.notebook
from scipy.special import softmax
import argparse

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda:0


In [2]:
# curl https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t36_3B_UR50D.pt -o esm2_t36_3B_UR50D.pt
# curl https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t48_15B_UR50D.pt -o esm2_t48_15B_UR50D.pt

#curl https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t48_15B_UR50D-contact-regression.pt -o esm2_t48_15B_UR50D-contact-regression.pt

# 1. Allow argparse.Namespace to be unpickled
torch.serialization.add_safe_globals([argparse.Namespace])

# 2. Define your path
model_path = "/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_proteome/00.esm_cache/esm2_t48_15B_UR50D.pt"

# 3. Load locally (this should now work)
model, alphabet = pretrained.load_model_and_alphabet_local(model_path)
batch_converter = alphabet.get_batch_converter()

model.eval()
model.to(DEVICE)
print("ESM-2 Loaded from local file.")

ESM-2 Loaded from local file.


In [4]:
def get_masked_marginal_scores(sequence, model, alphabet, batch_size=4):
    from torch.nn.functional import log_softmax

    model.eval()
    batch_converter = alphabet.get_batch_converter()
    device = next(model.parameters()).device

    standard_aa = list("ACDEFGHIKLMNPQRSTVWY")
    aa_indices = torch.tensor([alphabet.get_idx(a) for a in standard_aa]).to(device)

    # Tokenize WT once
    data = [("wt", sequence)]
    _, _, wt_tokens = batch_converter(data)
    wt_tokens = wt_tokens.to(device)

    seq_len = len(sequence)

    masked_tokens = []
    for i in range(seq_len):
        tokens = wt_tokens.clone()
        tokens[0, i + 1] = alphabet.mask_idx  # mask at token level
        masked_tokens.append(tokens)

    scores = []

    with torch.no_grad():
        for i in range(0, seq_len, batch_size):
            batch = torch.cat(masked_tokens[i:i+batch_size], dim=0)

            logits = model(batch)["logits"]
            log_probs = log_softmax(logits, dim=-1)

            for j in range(batch.size(0)):
                pos = i + j
                lp = log_probs[j, pos + 1, aa_indices]
                scores.append(lp.cpu().numpy())

    score_matrix = np.vstack(scores)

    final_scores = []
    for k in range(seq_len):
        wt_aa = sequence[k]
        wt_idx = standard_aa.index(wt_aa)
        final_scores.append(score_matrix[k] - score_matrix[k, wt_idx])

    df = pd.DataFrame(final_scores, columns=standard_aa)
    df.insert(0, "WT", list(sequence))
    df.insert(0, "Pos", range(1, seq_len + 1))

    return df

In [6]:
# 1. Define your sequence (Example: GFP)
grb2 = "MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDGFIPKNYIEMKPHPWFFGKIPRAKAEEMLSKQRHDGAFLIRESESAPGDFSLSVKFGNDVQHFKVLRDGAGKYFLWVVKFNSLNELVDYHRSTSVSRNQQIFLRDIEQVPQQPTYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores = get_masked_marginal_scores(grb2, model, alphabet, batch_size=2)

df_scores

,Pos,WT,A,C,D,E,F,G,H,I,...,M,N,P,Q,R,S,T,V,W,Y
0,1,M,-9.323103,-13.410257,-12.678099,-11.705435,-12.097483,-12.123864,-11.106535,-8.782368,...,0.000000,-10.887811,-9.609190,-10.349443,-10.122342,-10.826437,-9.606304,-8.386822,-13.708884,-12.355143
1,2,E,-7.542340,-12.869376,-5.050941,0.000000,-11.159039,-9.570625,-7.886359,-9.890934,...,-10.239139,-9.606495,-11.174699,-5.329512,-9.216734,-9.151344,-10.019423,-8.335674,-12.879145,-10.812786
2,3,A,0.000000,-6.752129,-11.530380,-11.895919,-9.396619,-6.571512,-11.665533,-9.424135,...,-9.962856,-9.260942,-9.364923,-12.489067,-11.951283,-1.874935,-5.570334,-6.348651,-11.531734,-9.426955
3,4,I,-2.706833,-2.454428,-6.653633,-5.104574,-1.815900,-6.162798,-3.046668,0.000000,...,-3.094535,-2.887734,-9.225142,-2.549223,-3.497420,-2.603880,-0.601645,-0.380022,-6.500099,-2.357989
4,5,A,0.000000,-7.586362,-8.811010,-9.115829,-11.896846,-0.780833,-10.663058,-11.369482,...,-11.775078,-10.127151,-9.933575,-10.982606,-9.612209,-5.729284,-7.003153,-7.799826,-12.347362,-11.332925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,213,V,-0.010397,0.762090,-4.968513,-5.006405,-0.911590,-2.646314,-2.994485,-0.648557,...,-3.298848,-3.151985,-4.993780,-4.338239,-3.036675,0.009218,-0.572172,0.000000,-2.546635,-0.294268
213,214,N,-1.239243,-3.054433,-0.797366,-1.221635,-2.710370,-1.923743,-1.277788,-1.892822,...,-2.596389,0.000000,-0.667609,-0.494322,-0.489374,-0.596931,-0.767976,-1.464039,-4.866656,-1.600835
214,215,R,-1.323455,-2.883107,-2.041406,-2.696976,-1.830569,-1.684757,-0.681223,-1.615261,...,-1.714229,0.700036,-0.637652,-1.215317,0.000000,0.079897,-0.764269,-1.375674,-4.654896,-1.277845
215,216,N,-0.384538,-1.445312,-0.827886,-1.088311,-0.594445,-0.093396,-0.678111,-1.091684,...,-1.731663,0.000000,-0.512722,-0.431717,-0.177692,0.112084,-0.586051,-0.874203,-2.416479,-0.342432


In [7]:
df_scores.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/grb2_esm2_15b_fl.csv", index=False)

In [8]:
# 1. Define your sequence (Example: GFP)
sh3 = "TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVN"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores2 = get_masked_marginal_scores(sh3, model, alphabet, batch_size=2)
df_scores2.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/grb2_sh3_esm2_15b_domain.csv", index=False)

In [ ]:
psd95 = "MDCLCIVTTKKYRYQDEDTPPLEHSPAHLPNQANSPPVIVNTDTLEAPGYELQVNGTEGEMEYEEITLERGNSGLGFSIAGGTDNPHIGDDPSIFITKIIPGGAAAQDGRLRVNDSILFVNEVDVREVTHSAAVEALKEAGSIVRLYVMRRKPPAEKVMEIKLIKGPKGLGFSIAGGVGNQHIPGDNSIYVTKIIEGGAAHKDGRLQIGDKILAVNSVGLEDVMHEDAVAALKNTYDVVYLKVAKPSNAYLSDSYAPPDITTSYSQHLDNEISHSSYLGTDYPTAMTPTSPRRYSPVAKDLLGEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYSRFEAKIHDLREQLMNSSLGSGTASLRSNPKRGFYIRALFDYDKTKDCGFLSQALSFRFGDVLHVIDASDEEWWQARRVHSDSETDDIGFIPSKRRVERREWSRLKAKDWGSSSGSQGREDSVLSYETVTQMEVHYARPIIILGPTKDRANDDLLSEFPDKFGSCVPHTTRPKREYEIDGRDYHFVSSREKMEKDIQAHKFIEAGQYNSHLYGTSVQSVREVAEQGKHCILDVSANAVRRLQAAHLHPIAIFIRPRSLENVLEINKRITEEQARKAFDRATKLEQEFTECFSAIVEGDSFEEIYHKVKRVIEDLSGPYIWVPARERL"

# 2. Run the calculation
# NOTE: batch_size=2 is a safe starting point for the 3B model
df_scores3 = get_masked_marginal_scores(psd95, model, alphabet, batch_size=2)
df_scores3.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/psd95_esm2_15b_fl.csv", index=False)

In [ ]:
psd95_pdz3 = "PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKP"

df_scores4 = get_masked_marginal_scores(psd95_pdz3, model, alphabet, batch_size=2)
df_scores4.to_csv("/lustre/scratch126/gengen/projects_v2/alpha-allostery-global/allostery_pathogenicity/git_allostery_pathogenicity/data/revision/esm2/psd95_pdz3_esm2_15b_fl.csv", index=False)